In [1]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv, find_dotenv

# 我们使用的代码是基于这篇文章的，是一个很好的介绍如何在python中实现REACT模式的文章
# based on https://til.simonwillison.net/llms/python-react-pattern
# Agent 由LLM + 围绕LLM构建的各种组件（可以统称为运行时）
_ = load_dotenv(find_dotenv())
from openai import OpenAI

In [2]:
client = OpenAI()


In [3]:
chat_completion = client.chat.completions.create(
    model="deepseek-ai/deepseek-v4-pro",
    messages=[{"role": "user", "content": "hello world"}],
)

print(chat_completion.choices[0].message.content)


Hello! How can I help you today?


In [4]:
 class Agent:
     def __init__(self, system=""):
         self.system = system
         self.messages = []
         if self.system:
             self.messages.append({"role":"system", "content": system})

     def __call__(self, message):
         self.messages.append({"role": "user", "content": message})
         result = self.execute()
         self.messages.append({"role": "assistant", "content": result})
         return result

     def execute(self):
         completion = client.chat.completions.create(
            model="deepseek-ai/deepseek-v4-pro",
             temperature=0,
            messages=self.messages,
        )
         return completion.choices[0].message.content
         

In [5]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you – then return PAUSE
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number – uses Python so be sure to use valid Python syntax

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [6]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier":
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [7]:
abot = Agent(prompt)

In [8]:
result = abot("How much does a toy poddle weigh")

In [9]:
print(result)

Thought: I need to find the average weight of a Toy Poodle using the average_dog_weight action. However, I should check if "toy poddle" is a typo and meant "Toy Poodle". I'll proceed with "Toy Poodle".
Action: average_dog_weight: Toy Poodle
PAUSE
